In [20]:
import json
import pandas as pd
import glob
import os


In [21]:

def generate_sdoh_table(json_filepaths):
    # The exact columns you requested
    columns = [
        'doc_id', 'Experiencer', 'financial_status', 'employment_status',
        'education_status', 'education_level', 'healthcare_type',
        'social_family_level', 'social_church_level', 'social_nonprofit_level',
        'social_government_level', 'mental_adhd', 'mental_ocd', 'mental_ptsd',
        'mental_anxiety', 'mental_depression', 'mental_bipolar',
        'mental_general', 'mental_medication', 'mental_neuro_condition',
        'mental_sleep_problem', 'smoke_status', 'substanceuse_status',
        'trauma_divorce', 'trauma_arrest', 'trauma_loss', 'trauma_other',
        'trauma_physical_abuse', 'trauma_psychological abuse',
        'trauma_domestic violence', 'trauma_dcf', 'trauma_abandonment',
        'insurance_type', 'adherence_medication', 'adherence_therapy',
        'adherence_other', 'transplant_knowledge', 'caregiving_knowledge',
        'medication_knowledge', 'increase_literacy', 'increase_social_support',
        'increase_financial_support', 'concern_level',
        'transportation_vehicle_access', 'transportation_cost',
        'transportation_distance', 'transportation_license',
        'transportation_violation', 'number of caregivers'
    ]
    
    # Potential keys that represent trauma sub-types in your JSONs
    trauma_keys = {
        'divorce': 'trauma_divorce', 'arrest': 'trauma_arrest', 'loss': 'trauma_loss',
        'other': 'trauma_other', 'physical_abuse': 'trauma_physical_abuse',
        'psychological abuse': 'trauma_psychological abuse', 
        'domestic violence': 'trauma_domestic violence', 'dcf': 'trauma_dcf',
        'abandonment': 'trauma_abandonment'
    }

    all_records = []

    for filepath in json_filepaths:
        # Extract doc_id from filename (e.g., "347" from "347_new_extracted.json")
        doc_id = os.path.basename(filepath).split('_')[0]
        
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        for item in data:
            predictions = item.get("extracted_predictions", {})
            for category, details in predictions.items():
                conditions = details.get("extracted_conditions", [])
                
                for condition in conditions:
                    experiencer = condition.get("Experiencer")
                    if not experiencer:
                        continue
                        
                    # Initialize empty record mapped to None
                    record = {col: None for col in columns}
                    record['doc_id'] = doc_id
                    record['Experiencer'] = experiencer
                    
                    # 1. Direct Column Mapping
                    for k, v in condition.items():
                        if k in columns:
                            record[k] = v
                            
                    # 2. Map Mental Health sub-types
                    if "mentalhealth_type" in condition:
                        mh_type = condition["mentalhealth_type"].lower().replace(" ", "_")
                        mh_col = f"mental_{mh_type}"
                        status = condition.get("mentalHealth_status", "yes") # default to yes if missing
                        if mh_col in columns:
                            record[mh_col] = status
                            
                    # 3. Map Trauma sub-types
                    for t_key, t_col in trauma_keys.items():
                        if t_key in condition:
                            record[t_col] = condition[t_key]
                            
                    all_records.append(record)

    # Convert to DataFrame
    df = pd.DataFrame(all_records)
    
    # Drop rows where we only have doc_id and Experiencer, but no actual data
    value_cols = [c for c in columns if c not in ['doc_id', 'Experiencer']]
    df = df.dropna(subset=value_cols, how='all')
    
    # --- The Crucial Step: Group by doc_id & Experiencer ---
    # .first() takes the first non-null value it encounters for each column in the group
    grouped_df = df.groupby(['doc_id', 'Experiencer'], dropna=False).first().reset_index()
    
    # Enforce exact column order
    grouped_df = grouped_df[columns]
    
    return grouped_df


In [28]:
json_files = glob.glob('../output/gpt4o/*_new_extracted.json')
final_table_gpt4o = generate_sdoh_table(json_files)
final_table_gpt4o.to_csv("../output/gpt4o/predict.csv")

In [29]:
json_files = glob.glob('../output/gpt55/*_new_extracted.json')
final_table_gpt55 = generate_sdoh_table(json_files)
final_table_gpt55.to_csv("../output/gpt55/predict.csv")

In [30]:
final_table_gpt4o

,doc_id,Experiencer,financial_status,employment_status,education_status,education_level,healthcare_type,social_family_level,social_church_level,social_nonprofit_level,...,increase_literacy,increase_social_support,increase_financial_support,concern_level,transportation_vehicle_access,transportation_cost,transportation_distance,transportation_license,transportation_violation,number of caregivers
0,100,Family,normal,None,None,None,hospital stay,high,low,None,...,None,None,None,None,easy,low,short,yes,no,None
1,100,Father,None,employed,None,None,None,None,None,None,...,None,None,None,None,easy,low,short,yes,no,None
2,100,Mother,None,on leave,None,None,None,None,None,None,...,None,None,None,None,easy,low,short,yes,no,None
3,100,children,None,None,current,childhood,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,100,parents/caregiver,None,employed,None,None,None,high,low,None,...,None,None,None,None,easy,low,short,yes,no,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227,99,Family,constrain,None,None,None,counseling,high,low,None,...,yes,no,no,None,None,None,None,None,None,None
228,99,Father,None,employed,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
229,99,Mother,None,on leave,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
230,99,parents/caregiver,None,None,None,None,medications,high,low,None,...,None,None,None,None,easy,low,short,yes,no,None
